In [0]:
%run ./08_Pipeline_Configurations

In [0]:
import uuid
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType, DoubleType
from pyspark.sql.functions import current_timestamp
from datetime import datetime

def write_audit_log(config_id, log_id, pipeline_name, layer_name, status, record_count, start_time, error_msg="None"):
    """
    Enhanced Audit Logging: Now includes pipeline_name for better filtering.
    """
    # 1. Calculate end time and duration
    end_time = datetime.now()
    duration_sec = (end_time - start_time).total_seconds()
    
    # 2. Schema definition (Updated with pipeline_name)
    schema = StructType([
        StructField("config_id", StringType(), True),
        StructField("log_id", StringType(), True),
        StructField("pipeline_name", StringType(), True),  # <--- Naya Column
        StructField("layer_name", StringType(), True),
        StructField("status", StringType(), True),
        StructField("record_count", IntegerType(), True),
        StructField("start_time", TimestampType(), True),
        StructField("end_time", TimestampType(), True),
        StructField("duration_sec", DoubleType(), True),
        StructField("error_message", StringType(), True)
    ])
    
    # 3. Data row
    log_data = [(config_id, log_id, pipeline_name, layer_name, status, record_count, start_time, end_time, duration_sec, error_msg)]
    
    # 4. Save to Table
    df_log = spark.createDataFrame(log_data, schema)
    df_log.write.format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable(AUDIT_LOG_TBL)
    
    print(f"✅ Logged: {pipeline_name} | {layer_name} | Status: {status} | Duration: {duration_sec}s")

print("✅ Enhanced Audit Log Function Loaded Successfully!")

In [0]:
import uuid
from datetime import datetime

# 1. Setup Tracking Variables
config_id = "BRONZE_CUST_INGEST"
log_id = str(uuid.uuid4())
pipeline_name = "Customer_Master_Pipeline"
start_time = datetime.now()

try:
    print("Starting Bronze Customer Ingestion...")
    
    # 2. Read raw data
    df_raw = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .load(RAW_CUSTOMER_PATH)
    
    record_count = df_raw.count()
    
    # 3. Write to Bronze layer with overwriteSchema fix
    df_raw.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(BRONZE_CUSTOMER_TBL)
    
    # 4. Success Log
    write_audit_log(config_id, log_id, pipeline_name, "Bronze_Customer", "SUCCESS", record_count, start_time)
    
    print("✅ Bronze Layer loaded successfully!")
    
except Exception as e:
    # 5. Failure Log (Error message captured correctly)
    error_msg = str(e)[:200]
    write_audit_log(config_id, log_id, pipeline_name, "Bronze_Customer", "FAILED", 0, start_time, error_msg)
    print(f"❌ Pipeline Failed: {error_msg}")
    raise e

In [0]:
import uuid
from datetime import datetime
from pyspark.sql.functions import col

# 1. Setup Tracking Variables
config_id = "SILVER_CUST_CLEANSE"
log_id = str(uuid.uuid4())
pipeline_name = "Customer_Master_Pipeline"
start_time = datetime.now()

try:
    print("Starting Silver Customer Cleansing...")
    
    # 2. Read from Bronze layer
    df_bronze = spark.read.table(BRONZE_CUSTOMER_TBL)
    
    # 3. Clean Data: Drop duplicates AND drop rows where critical info is null
    # Humne yahan customer_id, email aur phone teeno ko check kiya hai
    df_silver = df_bronze.dropDuplicates() \
                         .dropna(subset=["customer_id", "email", "phone"])
                         
    record_count = df_silver.count()
    
    # 4. Write to Silver layer with overwriteSchema fix
    df_silver.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(SILVER_CUSTOMER_TBL)
    
    # 5. Success Log
    write_audit_log(config_id, log_id, pipeline_name, "Silver_Customer", "SUCCESS", record_count, start_time)
    
    print("✅ Silver Customer Layer loaded successfully! (Null rows dropped)")
    
except Exception as e:
    # 6. Failure Log (Error message captured correctly)
    error_msg = str(e)[:200]
    write_audit_log(config_id, log_id, pipeline_name, "Silver_Customer", "FAILED", 0, start_time, error_msg)
    print(f"❌ Pipeline Failed: {error_msg}")
    raise e

In [0]:
import uuid
from datetime import datetime
from pyspark.sql.functions import col, when

# 1. Setup Tracking Variables
config_id = "GOLD_CUST_ENRICH"
log_id = str(uuid.uuid4())
pipeline_name = "Customer_Master_Pipeline"
start_time = datetime.now()

try:
    print("Starting Gold Customer Enrichment...")
    
    # 2. Read from Silver layer
    df_silver = spark.table(SILVER_CUSTOMER_TBL)
    
    # 3. Enrich Data (Business Logic)
    df_gold = df_silver.withColumn(
        "age_group",
        when(col("age") < 18, "Minor")
        .when((col("age") >= 18) & (col("age") <= 35), "Young Adult")
        .when((col("age") > 35) & (col("age") <= 55), "Adult")
        .otherwise("Senior")
    ).withColumn(
        "name_upper", col("name")
    ).select("customer_id", "name_upper", "country", "age_group", "email")
    
    record_count = df_gold.count()
    
    # 4. Write to Gold layer (Overwrite schema enabled)
    df_gold.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(GOLD_CUSTOMER_TBL)
    
    # 5. Success Log
    write_audit_log(config_id, log_id, pipeline_name, "Gold_Customer", "SUCCESS", record_count, start_time)
    print("✅ Gold Customer Layer enriched and loaded successfully!")
    
except Exception as e:
    # 6. Failure Log (Error message captured correctly)
    error_msg = str(e)[:200]
    write_audit_log(config_id, log_id, pipeline_name, "Gold_Customer", "FAILED", 0, start_time, error_msg)
    print(f"❌ Pipeline Failed: {error_msg}")
    raise e

In [0]:
import uuid
from datetime import datetime

# 1. Setup Tracking Variables
config_id = "BRONZE_ORDERS_INGEST"
log_id = str(uuid.uuid4())
pipeline_name = "Orders_Master_Pipeline"
start_time = datetime.now()

try:
    print("Starting Bronze Orders Ingestion...")
    
    # 2. Read raw data using variable from config
    df_raw = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .load(RAW_ORDERS_PATH)
    
    record_count = df_raw.count()
    
    # 3. Write to Bronze layer with overwriteSchema fix
    df_raw.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(BRONZE_ORDERS_TBL)
    
    # 4. Success Log (Updated with full parameters)
    write_audit_log(config_id, log_id, pipeline_name, "Bronze_Orders", "SUCCESS", record_count, start_time)
    
    print("✅ Bronze Orders Layer loaded successfully!")
    
except Exception as e:
    # 5. Failure Log (Error message captured correctly)
    error_msg = str(e)[:200]
    write_audit_log(config_id, log_id, pipeline_name, "Bronze_Orders", "FAILED", 0, start_time, error_msg)
    print(f"❌ Pipeline Failed: {error_msg}")
    raise e

In [0]:
import uuid
from datetime import datetime
from pyspark.sql.functions import col

# 1. Setup Tracking Variables
config_id = "SILVER_ORDERS_CLEANSE"
log_id = str(uuid.uuid4())
pipeline_name = "Orders_Master_Pipeline"
start_time = datetime.now()

try:
    print("Starting Silver Orders Cleansing...")
    
    # 2. Read from Bronze layer
    df_bronze = spark.table(BRONZE_ORDERS_TBL)
    
    # 3. Clean Data: Drop duplicates AND drop rows where critical order details are missing
    # Yahan hum order_id aur total_amount dono validate kar rahe hain
    df_silver = df_bronze.dropDuplicates() \
                         .dropna(subset=["order_id", "total_amount"])
    
    record_count = df_silver.count()
    
    # 4. Write to Silver layer with overwriteSchema fix
    df_silver.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(SILVER_ORDERS_TBL)
    
    # 5. Success Log
    write_audit_log(config_id, log_id, pipeline_name, "Silver_Orders", "SUCCESS", record_count, start_time)
    
    print("✅ Silver Orders Layer loaded successfully! (Incomplete orders dropped)")
    
except Exception as e:
    # 6. Failure Log (Error message captured correctly)
    error_msg = str(e)[:200]
    write_audit_log(config_id, log_id, pipeline_name, "Silver_Orders", "FAILED", 0, start_time, error_msg)
    print(f"❌ Pipeline Failed: {error_msg}")
    raise e

In [0]:
import uuid
from datetime import datetime
from pyspark.sql.functions import col, when, year, month

# 1. Setup Tracking Variables
config_id = "GOLD_ORDERS_ENRICH"
log_id = str(uuid.uuid4())
pipeline_name = "Orders_Master_Pipeline"
start_time = datetime.now()

try:
    print("Starting Gold Orders Enrichment...")
    
    # 2. Read from Silver layer
    df_silver = spark.table(SILVER_ORDERS_TBL)
    
    # 3. Enrich Data
    df_gold = df_silver \
        .withColumn("order_year", year(col("order_date"))) \
        .withColumn("order_month", month(col("order_date"))) \
        .withColumn("order_size", 
                    when(col("total_amount") >= 1000, "Large")
                    .when(col("total_amount") >= 500, "Medium")
                    .otherwise("Small")) \
        .withColumn("is_priority_order", 
                    when(col("status") == "Urgent", True).otherwise(False)) \
        .select("order_id", "customer_id", "total_amount", "order_date", 
                "order_year", "order_month", "order_size", "is_priority_order", "status")
    
    record_count = df_gold.count()
    
    # 4. Write to Gold layer (Fixed with overwriteSchema)
    df_gold.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(GOLD_ORDERS_TBL)
    
    # 5. Success Log
    write_audit_log(config_id, log_id, pipeline_name, "Gold_Orders", "SUCCESS", record_count, start_time)
    
    print("✅ Gold Orders Layer enriched and loaded successfully!")
    
except Exception as e:
    # 6. Failure Log (Error message captured correctly)
    error_msg = str(e)[:200]
    write_audit_log(config_id, log_id, pipeline_name, "Gold_Orders", "FAILED", 0, start_time, error_msg)
    print(f"❌ Pipeline Failed: {error_msg}")
    raise e

In [0]:
%sql
-- Audit Log
SELECT * 
FROM workspace.my_database.pipeline_audit_log
ORDER BY start_time DESC;